# Split CSV 분석

`SPLIT_CSV` 경로를 지정하면 pool 인코딩을 자동 감지하여 적절한 시각화를 수행합니다.

| pool 인코딩 | 설명 |
|---|---|
| ranked 1..N | entropy / anti_entropy — 전체 train subject BALD 순위 (1=최고 BALD) |
| binary 0/1 | random — pool 선택(1) / 미선택(0) |
| 없음 | pool 미사용 |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

SPLIT_CSV = "../checkpoints/stage2-v6-fedpid-entropy-0/agg/init/split.csv"

# ── 로드 ────────────────────────────────────────────────────────────────────
df = pd.read_csv(SPLIT_CSV)
df["Partition_ID"] = df["Partition_ID"].astype("Int64")
r_cols = sorted([c for c in df.columns if c.startswith("R") and c[1:].isdigit()])
for c in r_cols:
    df[c] = df[c].astype("Int64")
if "pool" in df.columns:
    df["pool"] = df["pool"].astype("Int64")

train_df = df[df["TrainOrVal"] == "train"].copy()
val_df   = df[df["TrainOrVal"] == "val"].copy()
partitions = sorted(train_df["Partition_ID"].dropna().unique())

# ── pool 인코딩 감지 ─────────────────────────────────────────────────────────
has_pool   = "pool" in df.columns
pool_max   = int(train_df["pool"].max(skipna=True)) if has_pool else 0
pool_zeros = int((train_df["pool"] == 0).sum()) if has_pool else 0
is_ranked  = has_pool and pool_max > 1 and pool_zeros == 0   # entropy/anti_entropy
is_binary  = has_pool and not is_ranked                       # random

pool_type = ("ranked 1..N  (entropy / anti_entropy)"
             if is_ranked else "binary 0/1  (random)"
             if is_binary else "없음")

print(f"CSV       : {SPLIT_CSV}")
print(f"전체 rows : {len(df)}  (train={len(train_df)}, val={len(val_df)})")
print(f"R 컬럼    : {r_cols}")
print(f"Pool 타입 : {pool_type}")
if is_ranked:
    print(f"  N={pool_max}  (전체 train subject BALD 순위 기록)")
elif is_binary:
    print(f"  pool=1: {int((train_df['pool']>0).sum())}  pool=0: {pool_zeros}")
df.head(10)

## Pool 분포

In [ ]:
if not has_pool:
    print("pool 컬럼 없음 — 생략")

elif is_ranked:
    # ── ranked pool: BALD 순위 분포 ─────────────────────────────────────────
    N = pool_max
    x = np.arange(len(partitions))
    xlabels = [f"P{p}" for p in partitions]

    has_r00 = "R00" in train_df.columns
    sel_df  = train_df[train_df["R00"] == 1] if has_r00 else None
    non_df  = train_df[train_df["R00"] == 0] if has_r00 else None

    if has_r00:
        fig, axes = plt.subplots(1, 2, figsize=(16, 4))

        # 왼쪽: R00 선택 / 비선택 stacked bar
        sel_cnt = sel_df.groupby("Partition_ID").size().reindex(partitions, fill_value=0)
        non_cnt = non_df.groupby("Partition_ID").size().reindex(partitions, fill_value=0)
        ax = axes[0]
        b1 = ax.bar(xlabels, sel_cnt.values, color="steelblue", edgecolor="white",
                    label=f"R00 selected ({int(sel_cnt.sum())})")
        ax.bar(xlabels, non_cnt.values, bottom=sel_cnt.values,
               color="lightgray", edgecolor="white",
               label=f"not selected ({int(non_cnt.sum())})")
        ax.bar_label(b1, labels=[str(v) if v > 0 else "" for v in sel_cnt.values],
                     label_type="center", fontsize=8, color="white")
        ax.set_title(f"R00 selected vs not  (total train={N})", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Subjects")
        ax.set_ylim(0, (sel_cnt + non_cnt).max() * 1.2)
        ax.legend(loc="upper right"); ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

        # 오른쪽: 선택/비선택 BALD 순위 중위수 grouped bar
        med_sel = sel_df.groupby("Partition_ID")["pool"].median().reindex(partitions)
        med_non = non_df.groupby("Partition_ID")["pool"].median().reindex(partitions)
        colors_sel = cm.RdYlGn_r(np.nan_to_num(med_sel.values) / N)
        w = 0.4
        ax = axes[1]
        bs = ax.bar(x - w/2, med_sel.values, width=w, color=colors_sel, edgecolor="white",
                    label="selected (median rank)")
        ax.bar(x + w/2, med_non.values, width=w, color="lightgray", edgecolor="white",
               label="not selected (median rank)")
        ax.bar_label(bs, labels=[f"{v:.0f}" if not np.isnan(v) else "" for v in med_sel.values],
                     padding=2, fontsize=7)
        ax.axhline(N / 2, color="gray", linestyle="--", linewidth=0.8, label=f"global median={N//2}")
        ax.set_title("Median BALD rank: selected vs not  (1=highest BALD)", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Median rank")
        ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=7)
        ax.set_ylim(0, N * 1.1)
        ax.legend(fontsize=8); ax.grid(axis="y", linestyle="--", alpha=0.4)

    else:
        # R00 없음: BALD 순위 분포만
        fig, ax = plt.subplots(figsize=(12, 4))
        medians = train_df.groupby("Partition_ID")["pool"].median().reindex(partitions)
        colors  = cm.RdYlGn_r(medians.values / N)
        bars = ax.bar(xlabels, medians.values, color=colors, edgecolor="white")
        ax.bar_label(bars, labels=[f"{v:.0f}" for v in medians.values], padding=3, fontsize=8)
        ax.axhline(N / 2, color="gray", linestyle="--", linewidth=0.8, label=f"global median={N//2}")
        ax.set_title(f"Median BALD rank per Partition  (N={N}, 1=highest BALD)", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Median rank")
        ax.set_ylim(0, N * 1.1)
        ax.legend(fontsize=9); ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

    plt.suptitle(SPLIT_CSV, fontsize=9)
    plt.tight_layout()
    plt.show()

    # 텍스트 테이블
    hdr = f"{'Partition':>12}  {'total':>6}  {'R00_sel':>8}  {'rank_min':>9}  {'rank_med_sel':>13}  {'rank_med_non':>13}  {'rank_max':>9}"
    print(hdr)
    print("-" * len(hdr))
    for p in partitions:
        pf = train_df[train_df["Partition_ID"] == p]
        sf = pf[pf["R00"] == 1] if has_r00 else pf
        nf = pf[pf["R00"] == 0] if has_r00 else None
        ms = f"{sf['pool'].median():.1f}" if len(sf) > 0 else "-"
        mn = f"{nf['pool'].median():.1f}" if (nf is not None and len(nf) > 0) else "-"
        print(f"{p:>12}  {len(pf):>6}  {len(sf):>8}  {int(pf['pool'].min()):>9}  {ms:>13}  {mn:>13}  {int(pf['pool'].max()):>9}")

else:
    # ── binary pool (random): pool>0 / pool=0 stacked bar ───────────────────
    all_cnt  = train_df.groupby("Partition_ID").size()
    pool_cnt = train_df[train_df["pool"] > 0].groupby("Partition_ID").size().reindex(all_cnt.index, fill_value=0)
    non_cnt  = all_cnt - pool_cnt
    xlabels  = [f"P{p}" for p in all_cnt.index]

    fig, ax = plt.subplots(figsize=(12, 3))
    b1 = ax.bar(xlabels, pool_cnt.values, color="steelblue", edgecolor="white",
                label=f"pool ({int(pool_cnt.sum())})")
    ax.bar(xlabels, non_cnt.values, bottom=pool_cnt.values,
           color="lightcoral", edgecolor="white", label=f"non-pool ({int(non_cnt.sum())})")
    ax.bar_label(b1, labels=[str(v) if v > 0 else "" for v in pool_cnt.values],
                 label_type="center", fontsize=8, color="white")
    ax.set_title(f"Pool membership per Partition  (total train={len(train_df)})", fontsize=11)
    ax.set_xlabel("Partition"); ax.set_ylabel("Subjects")
    ax.set_ylim(0, all_cnt.max() * 1.2)
    ax.legend(loc="upper right"); ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(axis="x", labelsize=7)
    plt.suptitle(SPLIT_CSV, fontsize=9)
    plt.tight_layout()
    plt.show()

## 라운드별 선택 분포 (R00 ~ Rnn)

In [ ]:
if not r_cols:
    print("R 컬럼 없음 — 생략")
else:
    # ranked pool → 전체 train 기준, binary → pool>0 기준, 없음 → 전체 train
    if is_ranked:
        plot_df    = train_df
        pool_label = f"all train (N={pool_max})"
    elif is_binary:
        plot_df    = train_df[train_df["pool"] > 0]
        pool_label = f"pool ({int((train_df['pool']>0).sum())})"
    else:
        plot_df    = train_df
        pool_label = f"all train ({len(train_df)})"

    fig, axes = plt.subplots(len(r_cols), 1, figsize=(12, 3 * len(r_cols)))
    if len(r_cols) == 1:
        axes = [axes]

    for ax, r_col in zip(axes, r_cols):
        sel      = plot_df.groupby("Partition_ID")[r_col].sum()
        total_s  = int(sel.sum())
        bars = ax.bar([f"P{p}" for p in sel.index], sel.values,
                      color="steelblue", edgecolor="white")
        ax.bar_label(bars, padding=4, fontsize=9)
        ax.set_title(f"{r_col}  (selected={total_s} / {pool_label})", fontsize=11)
        ax.set_xlabel("Partition"); ax.set_ylabel("Selected")
        ax.set_ylim(0, sel.max() * 1.3 if sel.max() > 0 else 1)
        ax.grid(axis="y", linestyle="--", alpha=0.4)
        ax.tick_params(axis="x", labelsize=7)

    fig.suptitle(SPLIT_CSV, fontsize=9)
    plt.tight_layout()
    plt.show()

    # 텍스트 테이블
    hdr = f"{'Partition':>12}  {'avail':>6}" + "".join(f"  {c:>8}" for c in r_cols)
    print(hdr)
    print("-" * len(hdr))
    for p in sorted(plot_df["Partition_ID"].dropna().unique()):
        pf = plot_df[plot_df["Partition_ID"] == p]
        row = f"{p:>12}  {len(pf):>6}"
        for c in r_cols:
            row += f"  {int(pf[c].sum()):>4}/{len(pf):<3}"
        print(row)

## BALD 순위 구간별 파티션 분포 (ranked pool 전용)

In [ ]:
if not is_ranked:
    print("ranked pool 아님 — 생략")
else:
    N      = pool_max
    N_BINS = 10
    bins   = np.linspace(1, N + 1, N_BINS + 1)
    labels = [f"{int(bins[i])}–{int(bins[i+1])-1}" for i in range(N_BINS)]
    train_df["rank_bin"] = pd.cut(train_df["pool"], bins=bins, labels=labels, right=False)

    # 구간 × 파티션 히트맵
    pivot = train_df.pivot_table(index="rank_bin", columns="Partition_ID",
                                  values="Subject_ID", aggfunc="count", fill_value=0)

    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"P{p}" for p in pivot.columns], fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    ax.set_xlabel("Partition"); ax.set_ylabel("BALD rank bin  (top → bottom)")
    ax.set_title(f"Subject count per rank bin × Partition  (N={N}, bin size≈{N//N_BINS})", fontsize=11)
    plt.colorbar(im, ax=ax, label="count")

    # 셀 값 표시
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = pivot.values[i, j]
            if v > 0:
                ax.text(j, i, str(v), ha="center", va="center", fontsize=7,
                        color="white" if v > pivot.values.max() * 0.6 else "black")

    plt.suptitle(SPLIT_CSV, fontsize=9)
    plt.tight_layout()
    plt.show()